In [8]:
import pandas as pd
import numpy as np

train = pd.read_parquet("../data/processed/train_ca1.parquet")
val = pd.read_parquet("../data/processed/val_ca1.parquet")

data = (
    pd.concat([train, val], axis=0)
    .sort_values(["item_id", "date"])
    .reset_index(drop=True)
)

data["date"] = pd.to_datetime(data["date"])
print(data.shape)

(5832737, 12)


In [9]:
# log features
for lag in [7, 14, 28]:
    data[f"lag_{lag}"] = data.groupby("item_id")["sales"].shift(lag)

In [10]:
# rolling means (7/28) without leakage
for window in [7, 28]:
    data[f"rolling_mean_{window}"] = (
        data.groupby("item_id")["sales"]
        .shift(1)
        .rolling(window)
        .mean()
        .reset_index(level=0, drop=True)
    )

In [11]:
# Price dynamics
data["price_lag_1"] = data.groupby("item_id")["sell_price"].shift(1)
data["price_change"] = data["sell_price"] - data["price_lag_1"]
data["price_change"] = data["price_change"].fillna(0)

In [12]:
# Time features
data["day_of_week"] = data["date"].dt.weekday
data["is_weekend"] = data["day_of_week"].isin([5, 6]).astype(int)

In [13]:
# This will remove the first ~28 days per item (expected).
feature_cols = [
    "sell_price", "wday", "month", "year", "snap_CA",
    "lag_7", "lag_14", "lag_28", "rolling_mean_7", "rolling_mean_28",
    "price_change", "day_of_week", "is_weekend"
]

data_fe = data.dropna(subset=["lag_28", "rolling_mean_28"]).copy()
print("After dropping NA for lags:", data_fe.shape)

After dropping NA for lags: (5747365, 21)


In [14]:
max_date = data["date"].max()
validation_start = max_date - pd.Timedelta(days=27)

train_fe = data_fe[data_fe["date"] < validation_start].copy()
val_fe = data_fe[data_fe["date"] >= validation_start].copy()

print("Train FE:", train_fe.shape)
print("Val FE:", val_fe.shape)

Train FE: (5661993, 21)
Val FE: (85372, 21)


In [15]:
train_fe.to_parquet("../data/processed/train_fe.parquet", index=False)
val_fe.to_parquet("../data/processed/val_fe.parquet", index=False)

print("Saved train_fe.parquet and val_fe.parquet")

Saved train_fe.parquet and val_fe.parquet


In [16]:
import pandas as pd

train_fe = pd.read_parquet("../data/processed/train_fe.parquet")
val_fe = pd.read_parquet("../data/processed/val_fe.parquet")

print("Train FE shape:", train_fe.shape)
print("Val FE shape:", val_fe.shape)

Train FE shape: (5661993, 21)
Val FE shape: (85372, 21)
